Import simulation outputs from LISFLOOD-FP and package as NetCDF files.

Ensure to specify `voutput` in LISFLOOD-FP parameters file to get velocity as well as water depth.

In [ ]:
import numpy as np
import os
import rioxarray as rxr
from typing import cast, Literal
import xarray

import graph_creation
from graph_creation import create_mesh_dhydro

LISFLOOD_OUTPUT_DIR = "/home/aidan/code/data/mSWE-GNN-train-test"
DEM_FILE = "/home/aidan/code/mSWE-GNN/database/raw_datasets_dyce/DEM/DEM_0.xyz"
PREFIX = "res_5m_acc_cuda"
MAX_STEP = 40

def read_step(step: int, prefix: str, ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"]|None=None) -> xarray.DataArray:
        return cast(xarray.DataArray, rxr.open_rasterio(os.path.join(LISFLOOD_OUTPUT_DIR, f"{prefix}-{int(step):04}.{ftype}"), parse_coordinates=True, masked=True))[0]

In [ ]:
def extract_parameter(ftype:Literal["wd", "wdfp", "elev", "Vx", "Vy"], max_step, prefix=PREFIX, shape=None):
    results = []
    for step in range(0, max_step+1):
        results.append(read_step(step, prefix, ftype))
    parameter_array = xarray.DataArray(results, dims=["time","x","y"])
    print(parameter_array)
    if shape:
        parameter_array = parameter_array.sel(x = slice(0, shape[0]), y = slice(0, shape[1]))
    return parameter_array

In [ ]:
DEM = []
with open(DEM_FILE) as f:
    for line in f:
        DEM.append([float(coord) for coord in line.split(" ")])
DEM = np.array(DEM)

In [ ]:
# import importlib
# importlib.reload(graph_creation)

In [ ]:
mesh = graph_creation.create_mesh_dhydro(DEM[:,:2], "raw_datasets_dyce/Geometry/polygon_0.pol", 1, False)[0]

In [ ]:
import matplotlib.pyplot as plt
%matplotlib qt
fig, ax = plt.subplots()
mesh.plot_edges(ax)
plt.show()

In [ ]:
# Find point source node for input boundary conditions
pointsource_x = 388911
pointsource_y = 814101
pointsource_idx = np.argmin((mesh.node_xy[:,0]-pointsource_x)**2 + (mesh.node_xy[:,1]-pointsource_y)**2)
print(mesh.node_xy[pointsource_idx,:2])
print("Distance: ", np.sqrt(np.sum((mesh.node_xy[pointsource_idx,:2] - np.array([pointsource_x, pointsource_y]))**2)) )

mesh.edge_type[mesh.edge_faces == pointsource_idx] = 2

In [ ]:
wd = extract_parameter("wd", MAX_STEP)
Vx = extract_parameter("Vx", MAX_STEP, shape=wd.shape[1:])
Vy = extract_parameter("Vy", MAX_STEP, shape=wd.shape[1:])

# node_x = 
# node_y = 
# face_x = DEM[:,0]
# face_y = DEM[:,1]
# edge_nodes = 
# edge_type = 
# edge_faces = 
# face_nodes = 

simulation_output = xarray.Dataset({
    "mesh2d_node_x": mesh.node_x,
    "mesh2d_node_y": mesh.node_y,
    "mesh2d_face_x": mesh.face_x,
    "mesh2d_face_y": mesh.face_y,
    "mesh2d_edge_nodes": mesh.edge_nodes,
    
    "mesh2d_edge_type": mesh.edge_type,
    "mesh2d_edge_faces": mesh.edge_faces,
    "mesh2d_face_nodes": mesh.face_nodes,

    "mesh2d_waterdepth": wd.stack(mesh2d_nFaces=("x","y")),
    "mesh2d_ucx": Vx.stack(mesh2d_nFaces=("x","y")),
    "mesh2d_ucy": Vy.stack(mesh2d_nFaces=("x","y"))
})

In [ ]:
simulation_output.reset_index("mesh2d_nFaces").to_netcdf("dyce_0.nc", format="NETCDF4")